# ⚽ Football Elo Dataset Processing Pipeline

This notebook takes your **pre-processed** results/Elo dataset and turns it into the
**final processed** dataset, then splits it into one CSV per country and packages
everything into a downloadable ZIP.

**Input columns (before this notebook runs):**
```
Date  Competition  Neutral Venue  Home  PreHElo  HEloChange  PostHElo  Result  Away  PreAElo  AEloChange  PostAElo
```

**Output columns (after this notebook runs):**
```
Country  Category  Date  Competition  Neutral Venue  Home  PreHElo  PostHElo  Result  Away  PreAElo  PostAElo  Home Goals  Away Goals
```

**What this notebook does, step by step:**
1. Forward-fill blank `Date` cells with the last seen date.
2. Map `Neutral Venue`: `'n' → 1`, blank → `0`.
3. Drop `HEloChange` and `AEloChange`.
4. Drop the 3 unnamed columns sitting between `Result` and `Away`.
5. Split `Result` (e.g. `2-1`) into `Home Goals` and `Away Goals`.
6. Use a reference file, `UEFA Competitions.csv`, to map `Competition → Country, Category`.
7. Split the final dataset into one CSV per `Country` (e.g. `Armenia.csv`, `Germany.csv`, ...).
8. Sort each country file by team name, then by date, and zip everything for download.

> Run the cells top to bottom. You will be prompted twice to upload a file:
> once for your main dataset, and once for `UEFA Competitions.csv`.


## Step 0 — Setup

Import everything we need. Nothing here needs to be edited.

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import zipfile
from google.colab import files

pd.set_option('display.max_columns', None)


## Step 0b — Upload your main dataset

Upload the **pre-processed** CSV (the one with columns
`Date, Competition, Neutral Venue, Home, PreHElo, HEloChange, PostHElo, Result, Away, PreAElo, AEloChange, PostAElo`,
possibly with a few extra unnamed columns).


In [ ]:
print("Please upload your main (pre-processed) results dataset...")
uploaded_main = files.upload()

main_filename = list(uploaded_main.keys())[0]
df = pd.read_csv(main_filename)

print(f"\nLoaded '{main_filename}' with shape {df.shape}")
print("\nColumns found in the file:")
print(list(df.columns))
df.head(10)


## Step 1 — Fill down the `Date` column

Whenever `Date` is blank, it belongs to the same date as the last non-blank value above it
(this is a common export artifact when a date only appears once per matchday block).

We treat empty strings and true NaNs the same way, then forward-fill.


In [ ]:
df['Date'] = df['Date'].replace(r'^\s*$', np.nan, regex=True)
df['Date'] = df['Date'].ffill()

# Sanity check: there should be no missing dates left
assert df['Date'].isna().sum() == 0, "There are still missing dates after forward-filling — check the top of the file."
print("Date column filled down. Remaining missing dates:", df['Date'].isna().sum())


## Step 2 — Map `Neutral Venue` to 0/1

The column only ever contains `'n'` or a blank value. We map:
- `'n'` → `1` (neutral venue)
- blank → `0` (not a neutral venue)

The code below is case/whitespace-tolerant, so `'N'`, `' n '`, etc. are also handled.


In [ ]:
df['Neutral Venue'] = (
    df['Neutral Venue']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
)

df['Neutral Venue'] = df['Neutral Venue'].map({'n': 1, '': 0})

# Anything that wasn't 'n' or blank would show up as NaN here — flag it instead of silently guessing
unmapped = df['Neutral Venue'].isna().sum()
if unmapped:
    print(f"Warning: {unmapped} rows in 'Neutral Venue' were neither 'n' nor blank. Inspect these rows.")

df['Neutral Venue'] = df['Neutral Venue'].fillna(0).astype(int)
print(df['Neutral Venue'].value_counts())


## Step 3 — Drop `HEloChange` and `AEloChange`

These are the raw single-match Elo deltas. We only need `PreHElo`/`PostHElo` and
`PreAElo`/`PostAElo`, so the change columns are dropped.


In [ ]:
df = df.drop(columns=['HEloChange', 'AEloChange'], errors='ignore')
print("Remaining columns:")
print(list(df.columns))


## Step 4 — Drop the 3 unnamed columns between `Result` and `Away`

When pandas reads a CSV with blank header cells, it auto-names them `Unnamed: N`.
The cell below finds every column sitting **between** `Result` and `Away` whose name
starts with `Unnamed`, and drops those.

It also prints what it's dropping and warns you if the count isn't 3, so you can double
check nothing unexpected is happening with your specific file.


In [ ]:
cols = list(df.columns)
result_idx = cols.index('Result')
away_idx = cols.index('Away')

between_cols = cols[result_idx + 1: away_idx]
unnamed_between = [c for c in between_cols if c.startswith('Unnamed')]

print("Columns found between 'Result' and 'Away':", between_cols)
print("Unnamed columns being dropped:", unnamed_between)

if len(unnamed_between) != 3:
    print(f"Warning: expected 3 unnamed columns between 'Result' and 'Away', found {len(unnamed_between)}. "
          f"Double-check 'between_cols' above before continuing.")

df = df.drop(columns=unnamed_between)
print("\nRemaining columns:")
print(list(df.columns))


## Step 5 — Split `Result` into `Home Goals` and `Away Goals`

`Result` stays in the final dataset as-is (e.g. `2-1`); we additionally extract the two
numbers either side of the dash into `Home Goals` and `Away Goals`.


In [ ]:
goals = df['Result'].astype(str).str.extract(r'(\d+)\s*-\s*(\d+)')

missing_goals = goals.isna().any(axis=1).sum()
if missing_goals:
    print(f"Warning: {missing_goals} rows didn't match the expected 'X-Y' pattern in 'Result'. Inspect these:")
    print(df.loc[goals.isna().any(axis=1), 'Result'].unique())

df['Home Goals'] = goals[0].astype('Int64')
df['Away Goals'] = goals[1].astype('Int64')

df[['Result', 'Home Goals', 'Away Goals']].head()


## Step 6 — Upload `UEFA Competitions.csv` and map `Competition → Country, Category`

Upload your reference file below. It's expected to contain at least the columns
`Competition`, `Country`, and `Category` (column names are stripped of stray whitespace
automatically). If your reference file uses slightly different column names, rename them
in the cell after the upload, before running the merge.


In [ ]:
print("Please upload 'UEFA Competitions.csv'...")
uploaded_ref = files.upload()

ref_filename = list(uploaded_ref.keys())[0]
ref = pd.read_csv(ref_filename)
ref.columns = [c.strip() for c in ref.columns]

print(f"\nLoaded '{ref_filename}' with shape {ref.shape}")
print("Columns:", list(ref.columns))
ref.head()


In [ ]:
# --- If your reference file's columns are named differently, rename them here, e.g.:
# ref = ref.rename(columns={'Comp': 'Competition', 'Nation': 'Country', 'Tier': 'Category'})

required_cols = {'Competition', 'Country', 'Category'}
missing = required_cols - set(ref.columns)
if missing:
    raise ValueError(
        f"Reference file is missing expected columns: {missing}. "
        f"Found columns: {list(ref.columns)}. Rename them in the cell above and re-run."
    )

df = df.merge(ref[['Competition', 'Country', 'Category']], on='Competition', how='left')

unmatched = sorted(df.loc[df['Country'].isna(), 'Competition'].unique())
if unmatched:
    print(f"Warning: {len(unmatched)} competition name(s) had no match in the reference table:")
    print(unmatched)
    print("These rows will have blank Country/Category unless you fix the reference file or the names.")
else:
    print("Every competition matched successfully.")


## Step 6b — Reorder columns to the final layout

```
Country  Category  Date  Competition  Neutral Venue  Home  PreHElo  PostHElo  Result  Away  PreAElo  PostAElo  Home Goals  Away Goals
```


In [ ]:
final_columns = [
    'Country', 'Category', 'Date', 'Competition', 'Neutral Venue', 'Home',
    'PreHElo', 'PostHElo', 'Result', 'Away', 'PreAElo', 'PostAElo',
    'Home Goals', 'Away Goals'
]

df = df[final_columns]
print(df.shape)
df.head(10)


## Step 7 & 8 — Sort, split by country, and zip

For each `Country`, we:
1. Sort rows by team name (`Home`) then by `Date`.
2. Write it out as `<Country>.csv`.

Then all the per-country CSVs are bundled into a single `processed_by_country.zip`,
which is downloaded automatically.

**Note on sorting:** "team name" is taken to mean the `Home` team, since each row
represents one match with a home and an away side rather than one row per team. If you'd
rather sort by the `Away` team instead, change `sort_by_team = 'Home'` to `'Away'` below.

**Note on dates:** `Date` is parsed assuming a day-first format (`DD/MM/YYYY`). If your
dates are month-first (`MM/DD/YYYY`), change `dayfirst=True` to `dayfirst=False` below.


In [ ]:
sort_by_team = 'Home'  # change to 'Away' if you'd rather sort by the away team
dayfirst = True        # change to False if your dates are MM/DD/YYYY

df['_Date_sort'] = pd.to_datetime(df['Date'], dayfirst=dayfirst, errors='coerce')

bad_dates = df['_Date_sort'].isna().sum()
if bad_dates:
    print(f"Warning: {bad_dates} rows had a 'Date' value that couldn't be parsed and will sort first.")

output_dir = 'processed_by_country'
os.makedirs(output_dir, exist_ok=True)

written_files = []
for country, group in df.groupby('Country', dropna=False):
    group_sorted = group.sort_values(by=[sort_by_team, '_Date_sort']).drop(columns=['_Date_sort'])

    country_label = 'Unknown' if pd.isna(country) else str(country)
    safe_name = re.sub(r'[^A-Za-z0-9_\-]', '_', country_label).strip('_') or 'Unknown'

    out_path = os.path.join(output_dir, f"{safe_name}.csv")
    group_sorted.to_csv(out_path, index=False)
    written_files.append(out_path)

print(f"Wrote {len(written_files)} country files:")
for f in written_files:
    print(" -", f)


In [ ]:
zip_path = 'processed_by_country.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_dir):
        zf.write(os.path.join(output_dir, fname), arcname=fname)

print(f"Created '{zip_path}' containing {len(os.listdir(output_dir))} files.")
files.download(zip_path)


## Done ✅

You should now have `processed_by_country.zip` downloaded to your computer, containing
one CSV per country (e.g. `Armenia.csv`, `Germany.csv`, ...), each sorted by team and
date, with the final column layout:

```
Country  Category  Date  Competition  Neutral Venue  Home  PreHElo  PostHElo  Result  Away  PreAElo  PostAElo  Home Goals  Away Goals
```
